This notebook is the blog / Chunk of my thoughts throughout the building process.

First I need to setup the pyproject.toml file, this file is a config file of what the project is and what resources we need.

Then I will make the basic structure of the whole project:

**notebooks folder:** annotate what I am doing throughout the project

**scripts folder:** data that I need for ML and sound analyze

**src/my_kws folder:**\
    `__init__.py`: The initial file\
    `augment.py`: data amplification\
    `dataset.py`: PyTorch Dataset\
    `features.py`: Extract characteristic\
    `io.py`: Audio I/O


Now here we have the code to only read audio file and output (mono information / sample rate / dtype)

In [ ]:
import soundfile as sf

def load_audio(
    path,
    target_sr,
    mono,
    target_length_s
):
    data, sr = sf.read(path)
    print(f"shape={data.shape}, sr={sr}, dtype={data.dtype}")
    return data

if __name__ == "__main__":
    x = load_audio(r"D:\Downloads\oboe_samples\oboe_samples\Ob-ord-A4-mf.wav", 16000, True, 1.0)

shape=(297437,), sr=44100, dtype=float64


**Explain:**\
We define the function load_audio.\
`path`: path of the audio file\
`target_sr`: the target sample rate\
`mono`: if this is mono audio file\
`target_length_s`: the target length of file loaded

**After we run the code, the result:**\
`shape=(297437,)`: this is the mono file, one dimentional.\
`sr=44100`: the sample rate is 44100Hz, we need to resample to 16000Hz.\
`dtype=float64`: we need to convert this to float32.\

In [ ]:
import numpy as np
import soundfile as sf
import torch
import torchaudio.functional as AF

def load_audio(
    path,
    target_sr,
    mono,
    target_length_s
):
    data, sr = sf.read(path)
    print(f"shape={data.shape}, sr={sr}, dtype={data.dtype}")

    #************************
    #convert to float32
    data = data.astype(np.float32)

    #resample (if sr != target_sr)
    if sr != target_sr:
        waveform = torch.from_numpy(data).unsqueeze(0)
        waveform = AF.resample(waveform, sr, target_sr)
        data = waveform.squeeze(0).numpy()
    #************************
    print(f"after resample: shape={data.shape}, sr={target_sr}, dtype={data.dtype}")
    return data

if __name__ == "__main__":
    x = load_audio(r"D:\learnMLDSP\edgeaudio-ml\data\raw\esc50\audio\1-100032-A-0.wav", 16000, True, 1.0)

shape=(220500,), sr=44100, dtype=float64
after resample: shape=(80000,), sr=16000, dtype=float32


**Now we have two more steps: convert dtype and resample.**

soundfile is a library that i/o audio files (WAV/FLAC/OGG), it is based on libsndfile.<br>
`sf.read(path)` return two things:<br>
    data: numpy array, which is the PCM(Pulse Code Modulation) of the sample value.<br>
    Unify to [-1.0, 1.0] float64<br>
    sr: sample rate (Usually 44.1kHz)<br>
    data shape: mono for 1D (N,) ; stereo for 2D (N,2)




numpy is a python library for numerical computing, all audio sample lists/matrix are ndarray.<br>

np.float32 is a dtype (single-precision floating-point)

**Why converting to float32?**<br>
    Pytorch/torchaudio need float32 by default.
    float64 takes double memory and is slower on GPU


`import torch`<br>
`torch.from_numpy(data)` -- converts numpy array into a Pytorch tensor with 0-copy (shared memory)<br>
`.unsqueeze(0)` -- adds a length-1D at position 0D (80000,) -> (1, 80000)<br>
`.squeeze(0)` -- remove the length-1D at position 0<br>
`.numpy` -- converts tensor back into a numpy array (shared memory)<br>


**Why unsqueeze?**<br>
torchaudio API take waveform with (channels, samples), so you have to have a channel dimension, so for mono you have to belike: (N,) -> (1, N)

`import torchaudio.functional as AF`<br>
torchaudio is the Official-audio library for PyTorch.<br>

`.functional` (F or AF) is the functional interface (stateless functions).<br>
`.transforms` is the counterpart, which provides stateful `nn.Module` classes.<br>
**Functional for one-offs because you pass everything in as arguments every call; Transforms for repeated use in a pipeline or model.**
`AF.resample(waveform, orig_sr, new_sr` is to resample. resample from `orig_sr` to `new_sr`.<br>

Normal audio file 44100Hz, KWS use 16000Hz (16kHz for Nyquist, save 2.75x data)

Use polyphase filter + lowpass to avoid aliasing.

**Polyphase filter**: mathematically rearranges the computation so you only ever compute the samples you actually keep. (Much faster)

In [ ]:
def load_audio(
        path,
        target_sr,
        mono,
        target_length_s
):
    data, sr = sf.read(path)
    print(f"shape={data.shape}, sr={sr}, dtype={data.dtype}")

    #convert to float32
    data = data.astype(np.float32)
    #*******************
    #check if mono
    if mono and data.ndim == 2:
        data = data.mean(axis=1)
    #*******************
    #resample (if sr != target_sr)
    if sr != target_sr:
        waveform = torch.from_numpy(data).unsqueeze(0) #shape: (1, n_samples)
        waveform = AF.resample(waveform, sr, target_sr)
        data = waveform.squeeze(0).numpy() #back to (n_samples,)

    print(f"after resample: shape={data.shape}, sr={target_sr}")
    
    return data

We make the audio mono if the audio is stereo.<br>
`if mono` -- means the user need mono as output, False if not.<br>
`data.ndim == 2` -- check if the data is 2D.<br>
`data.mean(axis=1)` -- find the mean of channel dimension.<br>

**axis=0**: reduces the first dimension / average along the "samples" dimension (Collapes N time steps into 1)<br>
**axis=1**: reduces the second dimension / average along the "channels" dimension (merges L and R channels into 1)<br>

In [ ]:
def load_audio(
        path,
        target_sr,
        mono,
        target_length_s
):
    data, sr = sf.read(path)
    print(f"shape={data.shape}, sr={sr}, dtype={data.dtype}")

    #convert to float32
    data = data.astype(np.float32)

    #check if mono
    if mono and data.ndim == 2:
        data = data.mean(axis=1)

    #resample (if sr != target_sr)
    if sr != target_sr:
        waveform = torch.from_numpy(data).unsqueeze(0) #shape: (1, n_samples)
        waveform = AF.resample(waveform, sr, target_sr)
        data = waveform.squeeze(0).numpy() #back to (n_samples,)
    print(f"after resample: shape={data.shape}, sr={target_sr}")
    #****************************
    #pad / crop
    target_n = int(target_sr * target_length_s)
    if len(data) > target_n:
        data = data[:target_n]
    elif len(data) < target_n:
        np.pad(data, (0, target_n - len(data)), mode='constant')
    #****************************
    return data

**Target Sample**: `target_n = int(target_sr * target_length_s)